This is a simple notebook demonstrating how you can use RF model for predicting e2etime for a single invocation

In [1]:
import joblib
import numpy as np
import json

In [2]:
path_to_rf_model = "../models/tuned_faas_rf_all_policies_proportionate.joblib"

In [3]:
# 1. Load the RF model
rf_model = joblib.load(path_to_rf_model)

In [4]:
# Number of features passed to the model
rf_model.n_features_in_

18

In [5]:
#Let's see the features we are providing
rf_model.feature_names_in_

array(['target_queue_len', 'others_len_queue', 'is_status_Active',
       'is_status_Inactive', 'is_status_Throttled', 'iat', 'iat_fqdn',
       'num_running_funcs_filled', 'gpu_warm_results_sec',
       'gpu_cold_results_sec', 'is_cold_start', 'policy_AlwaysGPU',
       'policy_Greedy', 'policy_Landlord', 'policy_MICE',
       'policy_NEW_MICE', 'policy_Speedup', 'policy_WeightedRandom_0.5'],
      dtype=object)

In [6]:
def get_fqdn_representation(fqdn, json_path='/Users/akshaykishan/PycharmProjects/iluvatar-faas/S4forIluvatar/src/data/worker_function_benchmarks.json'):
    """
    Reads the benchmark JSON and maps warm and cold execution times to the fqdn via base_function.
    """
    try:
        with open(json_path, 'r') as f:
            benchmarks = json.load(f)
    except FileNotFoundError:
        print(f"Warning: {json_path} not found. Giving benchmark features 0.0.")
        return 0.0, 0.0
    
    import re
    base_function = re.search(r'^([^0-9]+)', fqdn).group(1).strip('-')      

    data = benchmarks.get('data', {})
    warm_mean, cold_mean = 0.0, 0.0
    if base_function in data.keys():
        try:
            gpu_data = data[base_function].get('resource_data', {}).get('gpu', {})
            warm_mean = np.mean(gpu_data.get('warm_results_sec', [0]))
            cold_mean = np.mean(gpu_data.get('cold_results_sec', [0]))
        except Exception:
            warm_mean, cold_mean = 0.0, 0.0
            
    return warm_mean, cold_mean

In [7]:
# Sample data
fqdn = "cupy-4-0.0.1"

# numerical representation for fqdn from historical data
gpu_warm_results_sec, gpu_cold_results_sec = get_fqdn_representation(fqdn)

# Queue representation
target_queue_len = 0.0
others_len_queue = 0.0

#Target queue status
is_status_Active = 0
is_status_Inactive = 1
is_status_Throttled = 0

iat = 0.0
iat_fqdn = 0.0

#GPU representation
num_running_funcs_filled = 0.0
is_cold_start = 0 #Binary flag

#Dispatching Policy Indicator features(one-hot encoding)
policy_AlwaysGPu = 0
policy_Greedy = 0 
policy_Landlord = 1 #Data from landlord policy
policy_MICE = 0
policy_NEW_MICE = 0
policy_Speedup = 0
policy_WeightedRandom = 0

In [42]:
#features should be passed to the model in the same order as it was trained on

In [8]:
rf_model.feature_names_in_

array(['target_queue_len', 'others_len_queue', 'is_status_Active',
       'is_status_Inactive', 'is_status_Throttled', 'iat', 'iat_fqdn',
       'num_running_funcs_filled', 'gpu_warm_results_sec',
       'gpu_cold_results_sec', 'is_cold_start', 'policy_AlwaysGPU',
       'policy_Greedy', 'policy_Landlord', 'policy_MICE',
       'policy_NEW_MICE', 'policy_Speedup', 'policy_WeightedRandom_0.5'],
      dtype=object)

In [9]:
datapoint = np.array([target_queue_len, others_len_queue, is_status_Active,is_status_Inactive,  is_status_Throttled, 
iat, iat_fqdn, num_running_funcs_filled, gpu_warm_results_sec, gpu_cold_results_sec, is_cold_start, policy_AlwaysGPu, 
policy_Greedy, policy_Landlord, policy_MICE, policy_NEW_MICE, policy_Speedup, policy_WeightedRandom
])

In [12]:
# The RF model is trained after transforming e2etime to log-scale. Hence, rescaling it back to original by using exponentials
predicted_time = np.expm1(rf_model.predict(datapoint.reshape(1, -1)))

/opt/miniconda3/lib/python3.12/site-packages/sklearn/base.py:464: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [13]:
predicted_time

array([1.98971173])